# 06 - Hyperparameter Tuning

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = BASE_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
REPORTS = BASE_DIR / "reports"
FIGURES = REPORTS / "figures"
MODELS = BASE_DIR / "models"

print("Project:", BASE_DIR)
print("Artifacts present:", REPORTS.exists(), FIGURES.exists(), MODELS.exists())

Project: C:\Users\Province Settat\Desktop\ML\project
Artifacts present: True True True


Summarize the class-weighted and cross-validated training strategy used by the pipeline.

In [2]:
df = pd.read_csv(DATA_RAW / "tox21.csv")
y = df["NR-AhR"].dropna().astype(int).to_numpy()
n_pos = int(y.sum())
n_neg = int((y == 0).sum())
class_weights = {
    0: len(y) / (2 * n_neg),
    1: len(y) / (2 * n_pos),
}
print("NR-AhR labeled:", len(y))
print("Positive:", n_pos, "Negative:", n_neg)
print("Balanced class weights:", class_weights)
print("Pipeline uses stratified splitting and 5-fold CV for primary Morgan models.")

NR-AhR labeled: 6549
Positive: 768 Negative: 5781
Balanced class weights: {0: 0.5664244940321743, 1: 4.263671875}
Pipeline uses stratified splitting and 5-fold CV for primary Morgan models.


In [3]:
with open(REPORTS / "NR_AhR_morgan_metrics.json") as f:
    metrics = json.load(f)
cv_rows = []
for name, m in metrics.items():
    cv_rows.append({
        "model": name,
        "cv_f1_mean": m.get("cv_f1_mean"),
        "cv_f1_std": m.get("cv_f1_std"),
        "cv_roc_auc_mean": m.get("cv_roc_auc_mean"),
        "cv_roc_auc_std": m.get("cv_roc_auc_std"),
    })
display(pd.DataFrame(cv_rows).dropna(how="all", subset=["cv_f1_mean", "cv_roc_auc_mean"]))

,model,cv_f1_mean,cv_f1_std,cv_roc_auc_mean,cv_roc_auc_std
0,LogisticRegression,0.541221,0.035294,0.857020,0.018644
1,DecisionTree,0.371988,0.022945,0.759226,0.022565
2,RandomForest,0.430708,0.041294,0.891979,0.011394
3,XGBoost,0.153391,0.034952,0.875129,0.007934
4,LightGBM,0.539218,0.032501,0.873098,0.011236
5,SVM,0.557262,0.049788,0.891642,0.011861
